# Entrenamiento del modelo de riesgo crediticio

In [41]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import joblib

In [42]:
#Cargue el CSV en un DataFrame.
df = pd.read_csv('data/credit_score_raw.csv', low_memory=False)
df.shape

(100000, 28)

In [43]:
df.dtypes

ID                              str
Customer_ID                     str
Month                           str
Name                            str
Age                             str
SSN                             str
Occupation                      str
Annual_Income                   str
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                     str
Type_of_Loan                    str
Delay_from_due_date           int64
Num_of_Delayed_Payment          str
Changed_Credit_Limit            str
Num_Credit_Inquiries        float64
Credit_Mix                      str
Outstanding_Debt                str
Credit_Utilization_Ratio    float64
Credit_History_Age              str
Payment_of_Min_Amount           str
Total_EMI_per_month         float64
Amount_invested_monthly         str
Payment_Behaviour               str
Monthly_Balance                 str
Credit_Score                

In [44]:
df.isnull().sum()

ID                              0
Customer_ID                     0
Month                           0
Name                         9985
Age                             0
SSN                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit            0
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           9030
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      4479
Payment_Behaviour               0
Monthly_Balance              1200
Credit_Score                    0
dtype: int64

## Limpieza de datos

In [45]:
columnas_sucias = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
                   'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly',
                   'Monthly_Balance']

for col in columnas_sucias:
    df[col] = df[col].astype(str).str.replace('_', '', regex=False).str.strip()
    df[col] = df[col].replace({'': np.nan, 'nan': np.nan})
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [46]:
extraido = df['Credit_History_Age'].astype(str).str.extract(r'(\d+)\s*Years?\s*and\s*(\d+)\s*Months?')
df['Credit_History_Age_Meses'] = pd.to_numeric(extraido[0], errors='coerce') * 12 + pd.to_numeric(extraido[1], errors='coerce')

In [47]:
#Columnas numericas candidatas a ser features (se excluyen IDs, nombres, fechas y texto libre)
columnas_numericas = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
                      'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
                      'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
                      'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age_Meses',
                      'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

len(columnas_numericas)

17

## Completar valores nulos

In [48]:
for col in columnas_numericas:
    df[col] = df[col].fillna(df[col].median())

df[columnas_numericas].isnull().sum()

Age                         0
Annual_Income               0
Monthly_Inhand_Salary       0
Num_Bank_Accounts           0
Num_Credit_Card             0
Interest_Rate               0
Num_of_Loan                 0
Delay_from_due_date         0
Num_of_Delayed_Payment      0
Changed_Credit_Limit        0
Num_Credit_Inquiries        0
Outstanding_Debt            0
Credit_Utilization_Ratio    0
Credit_History_Age_Meses    0
Total_EMI_per_month         0
Amount_invested_monthly     0
Monthly_Balance             0
dtype: int64

## Variable objetivo

In [49]:
df['Credit_Score'].unique()

<StringArray>
['Good', 'Standard', 'Poor']
Length: 3, dtype: str

In [50]:
df['target'] = df['Credit_Score'].map({
    'Good': 0,
    'Standard': 1,
    'Poor': 2
})

df['target'].value_counts()

target
1    53174
2    28998
0    17828
Name: count, dtype: int64

## Eliminacion de valores atipicos (metodo IQR)

In [51]:
print('Registros antes de IQR:', df.shape)

for col in columnas_numericas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    df = df[(df[col] >= limite_inferior) & (df[col] <= limite_superior)]

print('Registros despues de IQR:', df.shape)

Registros antes de IQR: (100000, 30)
Registros despues de IQR: (56312, 30)


## Seleccion de las 5 caracteristicas mas influyentes

In [52]:
correlaciones = df[columnas_numericas + ['target']].corr()['target'].drop('target').abs()
correlaciones = correlaciones.sort_values(ascending=False)
print(correlaciones)

Interest_Rate               0.501192
Num_Credit_Inquiries        0.444357
Outstanding_Debt            0.428624
Delay_from_due_date         0.427823
Num_Credit_Card             0.400285
Num_Bank_Accounts           0.378862
Credit_History_Age_Meses    0.373163
Num_of_Loan                 0.346308
Num_of_Delayed_Payment      0.342033
Monthly_Balance             0.183823
Age                         0.160074
Changed_Credit_Limit        0.154554
Total_EMI_per_month         0.122962
Annual_Income               0.115972
Monthly_Inhand_Salary       0.104666
Amount_invested_monthly     0.060079
Credit_Utilization_Ratio    0.022570
Name: target, dtype: float64


In [53]:
features = correlaciones.head(5).index.tolist()
features

['Interest_Rate',
 'Num_Credit_Inquiries',
 'Outstanding_Debt',
 'Delay_from_due_date',
 'Num_Credit_Card']

## Division en entrenamiento y prueba, y escalado estandar

In [54]:
X = df[features]
y = df['target']
#agregué el stratify por el desbalance de clases y para que los resultados sean reproducibles.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Entrenamiento del modelo KNN

In [55]:
valores_k = [3, 5, 7, 9, 11, 15]
resultados_k = []

for k in valores_k:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    resultados_k.append(acc)
    print('k =', k, '-> accuracy =', acc)

k = 3 -> accuracy = 0.7444730533605611
k = 5 -> accuracy = 0.7371037911746426
k = 7 -> accuracy = 0.7247624966705141
k = 9 -> accuracy = 0.7179259522329753
k = 11 -> accuracy = 0.7165053715706295
k = 15 -> accuracy = 0.7114445529610228


In [56]:
k_optimo = valores_k[resultados_k.index(max(resultados_k))]
k_optimo #Se elige el k con mayor accuracy sobre el conjunto de prueba

3

In [57]:
knn = KNeighborsClassifier(n_neighbors=k_optimo)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

## Evaluacion del modelo

In [58]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print('Accuracy:', accuracy)
print('F1-score:', f1)

Accuracy: 0.7444730533605611
F1-score: 0.7225433780465768


In [59]:
print(classification_report(y_test, y_pred, target_names=['0 (bajo)', '1 (medio)', '2 (alto)']))

              precision    recall  f1-score   support

    0 (bajo)       0.64      0.66      0.65      1837
   1 (medio)       0.78      0.77      0.77      6096
    2 (alto)       0.74      0.75      0.75      3330

    accuracy                           0.74     11263
   macro avg       0.72      0.73      0.72     11263
weighted avg       0.75      0.74      0.74     11263



## Guardado del modelo entrenado

In [60]:
joblib.dump(knn, 'model/knn_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')

with open('model/features.json', 'w', encoding='utf-8') as f:
    json.dump({'features': features, 'k': k_optimo}, f, ensure_ascii=False, indent=2)